# IR Assignment 4: Email Spam Filtering using KNN

In [1]:
%pip install -q pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import urllib.request
import pandas as pd

# Reliable, active URL for Kaggle's SMS Spam Collection dataset (spam.csv)
url = "https://raw.githubusercontent.com/mohitgupta-omg/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv"
target_path = "spam.csv"

try:
    print(f"Downloading dataset from {url}...")
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response, open(target_path, 'wb') as out_file:
        out_file.write(response.read())
    print("Download complete! Saved as 'spam.csv'.")

    # Verify file loading
    df = pd.read_csv(target_path, encoding='latin-1')
    print("Successfully loaded dataset. Shape:", df.shape)
    print(df.head(2))
except Exception as e:
    print(f"Error downloading dataset: {e}")

Download complete! Saved as 'spam.csv'.
Successfully loaded dataset. Shape: (5572, 5)
    v1                                                 v2 Unnamed: 2  \
0  ham  Go until jurong point, crazy.. Available only ...        NaN   
1  ham                      Ok lar... Joking wif u oni...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  


In [3]:
# Assignment 4: Email Spam Filtering using KNN
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------------
# STEP 1: Load the dataset
# ------------------------------------------------------------------
# Keep spam.csv in the same folder as this Python file.
data = pd.read_csv("spam.csv", encoding="latin-1")

# Standardize column names
data.columns = data.columns.str.strip().str.lower()

# Support common dataset column names
if "v1" in data.columns and "v2" in data.columns:
    data = data.rename(columns={"v1": "label", "v2": "message"})
elif "category" in data.columns and "message" in data.columns:
    data = data.rename(columns={"category": "label"})
elif "label" not in data.columns or "message" not in data.columns:
    raise ValueError(
        "Expected label/message columns or v1/v2 columns. "
        f"Found: {data.columns.tolist()}"
    )

data = data[["label", "message"]].dropna()

# Normalize labels and remove unexpected classes
data["label"] = data["label"].astype(str).str.strip().str.lower()
data = data[data["label"].isin(["ham", "spam"])].copy()
data["message"] = data["message"].astype(str)

print("First five messages:")
print(data.head())
print("\nDataset size:", data.shape)
print("\nClass counts:")
print(data["label"].value_counts())

# ------------------------------------------------------------------
# STEP 2: Separate text and labels
# ------------------------------------------------------------------
X = data["message"]
y = data["label"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ------------------------------------------------------------------
# STEP 3: Convert text into numerical features
# ------------------------------------------------------------------
# TF-IDF gives importance to words in each message.
# The vectorizer is fitted only on the training data.

# ------------------------------------------------------------------
# STEP 4: Train the KNN classifier
# ------------------------------------------------------------------
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english"
    )),
    ("knn", KNeighborsClassifier(
        n_neighbors=5,
        metric="cosine",
        algorithm="brute",
        weights="distance"
    ))
])

model.fit(X_train, y_train)
print("\nKNN model training completed.")

# ------------------------------------------------------------------
# STEP 5: Predict on unseen test messages
# ------------------------------------------------------------------
y_pred = model.predict(X_test)
print("\nActual labels: ")
print(y_test.head(10).tolist())
print("\nPredicted labels:")
print(y_pred[:10].tolist())

# ------------------------------------------------------------------
# STEP 6: Evaluate model performance
# ------------------------------------------------------------------
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test, y_pred, pos_label="spam", zero_division=0
)
recall = recall_score(
    y_test, y_pred, pos_label="spam", zero_division=0
)

print("\nModel Performance")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall : {recall:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# Fix the class order so the confusion matrix is predictable
cm = confusion_matrix(y_test, y_pred, labels=["ham", "spam"])
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix (rows = actual, columns = predicted)")
print("                Predicted Ham    Predicted Spam")
print(f"Actual Ham      {tn:5d}            {fp:5d}")
print(f"Actual Spam     {fn:5d}            {tp:5d}")

specificity = tn / (tn + fp) if (tn + fp) else 0.0
print(f"\nSpecificity: {specificity:.4f}")

# ------------------------------------------------------------------
# STEP 7: Test a new message
# ------------------------------------------------------------------
new_message = [
    "Congratulations! You have won a free prize. Claim now!"
]
prediction = model.predict(new_message)[0]

print("\nNew message:", new_message[0])
print("Predicted category:", prediction.upper())

# ------------------------------------------------------------------
# STEP 8: Conclusion
# ------------------------------------------------------------------
print("\nConclusion:")
print("Spam filtering using TF-IDF and KNN is completed.")

First five messages:
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Dataset size: (5572, 2)

Class counts:
label
ham     4825
spam     747
Name: count, dtype: int64

KNN model training completed.

Actual labels: 
['ham', 'ham', 'ham', 'spam', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham']

Predicted labels:
['ham', 'ham', 'ham', 'spam', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham']

Model Performance
Accuracy: 0.9695
Precision: 0.9915
Recall : 0.7785

Classification Report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       0.99      0.78      0.87       149

    accuracy                           0.97      1115
   m